# Toolbox: comunicação coletiva no PyTorch

Aula 3 - `torch.distributed` é a **caixa de ferramentas**, não um jeito de treinar.
O DDP é quem as usa no `backward()`. Não há rede neural aqui: só tensores no process group.

## Por que a toolbox existe

`mp` sobe processos; sem troca de tensores, você fica com N treinos. Estas ops
viram **um** treino (isso é o DDP).

| ferramenta | o que faz | no DDP |
|---|---|---|
## Por que a toolbox existe

`mp` sobe processos; sem troca de tensores, você fica com N treinos. Estas ops
viram **um** treino (isso é o DDP).

| ferramenta | o que faz | no DDP |
|---|---|---|
| `all_reduce` | todos ficam com a soma | gradientes |
| `broadcast` | rank 0 copia para os outros | pesos iniciais |
| `scatter` | rank 0 parte a lista; cada um recebe um pedaço | fatiar batch / embeddings |
| `gather` | cada um manda o seu; só o dst recebe a lista | juntar métricas num rank |
| `all_gather` | cada um manda o seu; todos recebem a lista | métricas |
| `barrier` | esperam no mesmo ponto | download |

# 0. Instalando

Só o PyTorch. `gloo` já vem junto; `nccl` entra quando o treino é em GPU.

In [1]:
%pip install -q torch

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

`dist` = toolbox. `mp` = criar processos (no `torchrun` o launcher faz isso).

In [2]:
import os

import torch
import torch.distributed as dist
import torch.multiprocessing as mp

## 2. Process group

`world_size` = quantos. `rank` = id. `init_process_group` = ponto de encontro.
`gloo` = CPU (aula). `nccl` = GPU.

In [3]:
def init(rank, world_size, port="29511"):
    """Entra no process group. Sem isso, all_reduce e broadcast travam."""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = port
    dist.init_process_group(backend="gloo", rank=rank, world_size=world_size)
    print(f"Rank {rank}/{dist.get_world_size()}")

## 3. all_reduce

Todos terminam com a soma: `[1,1]` e `[2,2]` viram `[3,3]` nos dois ranks.
No DDP isso acontece com o **gradiente**.

In [4]:
def all_reduce_demo(rank):
    """Todos somam; todos ficam com o resultado. No DDP isso é o gradiente."""
    tensor = torch.ones(2) * (rank + 1)
    print(rank, "antes ", tensor.tolist())
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    print(rank, "depois", tensor.tolist())

## 4. broadcast

Rank 0 envia `[42, 7]`; os outros recebem. No DDP: pesos iniciais.

In [5]:
def broadcast_demo(rank):
    """Rank 0 copia o tensor para os outros. No DDP: pesos iniciais."""
    payload = torch.tensor([42.0, 7.0]) if rank == 0 else torch.zeros(2)
    dist.broadcast(payload, src=0)
    print(rank, "broadcast", payload.tolist())

## 5. scatter

Rank 0 tem a lista; cada rank recebe **um** pedaço. Inverso do `gather`.
Só a `def` - não entra no worker.

In [ ]:
def scatter_demo(rank):
    """Rank 0 parte a lista; cada rank fica com um pedaço."""
    out = torch.zeros(2)
    chunks = [torch.tensor([1.0, 1.0]), torch.tensor([2.0, 2.0])] if rank == 0 else None
    dist.scatter(out, scatter_list=chunks, src=0)
    print(rank, "scatter", out.tolist())

## 6. gather

Cada rank manda o seu; **só o dst** (rank 0) fica com a lista.
No `all_gather` todo mundo recebe. Só a `def` - não entra no worker.

In [ ]:
def gather_demo(rank):
    """Cada rank manda o seu; só o dst recebe a lista."""
    piece = torch.ones(2) * (rank + 1)
    collected = [torch.zeros(2), torch.zeros(2)] if rank == 0 else None
    dist.gather(piece, gather_list=collected, dst=0)
    print(rank, "gather", None if collected is None else [t.tolist() for t in collected])